# Bangladeshi Banknote Recognition using Digital Image Processing and Machine Learning
### CSE 4383 — Computer Vision and Image Processing | IUBAT

This notebook implements the full pipeline required by the assignment:

1. Dataset acquisition & exploration
2. Image preprocessing (denoising, normalization, edge detection, segmentation)
3. Note localization (classical CV) → enables **IoU** evaluation
4. Feature extraction (HOG, color histograms)
5. Classical ML model (SVM / Random Forest) on handcrafted features
6. Deep Learning model (CNN, transfer learning) end-to-end
7. Evaluation: Accuracy, Precision, Recall, F1, IoU, Confusion Matrix
8. Explainability (Grad-CAM)
9. Classical vs Deep Learning comparison
10. Notes for the Ethics section of the report

**Run cells top to bottom.** Runtime: `Runtime > Change runtime type > GPU` (T4 is fine).


In [ ]:
# 1. Install / import packages
!pip install -q opencv-python-headless scikit-image scikit-learn tensorflow kaggle

import os, json, glob, random, shutil, time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report)
import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__, "| GPU available:", tf.config.list_physical_devices('GPU'))


## Step 1 — Get the dataset

Upload your `kaggle.json` API token (Kaggle account → Settings → Create New Token),
then run the cell below. It downloads and unzips the dataset into `/content/data`.

If your Kaggle folder structure differs from `data/<denomination>/*.jpg`, adjust
`DATA_DIR` below after inspecting the unzip output.


In [ ]:
from google.colab import files
uploaded = files.upload()  # select kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d rahnumatasnim1604103/bangladeshi-banknote-dataset -p /content/data --unzip

DATA_DIR = '/content/data'  # <-- update this if the unzip creates a nested folder
print(os.listdir(DATA_DIR))


## Step 2 — Explore the dataset

Count images per class and look at samples. This also feeds directly into your
report's **Dataset** section (class balance, image quality, variation in
background/lighting — mention these as real-world challenges).


In [ ]:
def find_class_dirs(root):
    # Handles either root/<class>/*.jpg or root/<nested>/<class>/*.jpg
    candidates = [d for d in glob.glob(os.path.join(root, '**'), recursive=True)
                  if os.path.isdir(d) and len(glob.glob(os.path.join(d, '*.*'))) > 0]
    return candidates

class_dirs = sorted([d for d in glob.glob(os.path.join(DATA_DIR, '*')) if os.path.isdir(d)])
print("Detected class folders:", [os.path.basename(d) for d in class_dirs])

counts = {os.path.basename(d): len(glob.glob(os.path.join(d, '*.*'))) for d in class_dirs}
print(counts)

plt.figure(figsize=(8,4))
plt.bar(counts.keys(), counts.values())
plt.title("Images per denomination class")
plt.ylabel("count"); plt.xticks(rotation=45)
plt.tight_layout(); plt.savefig('/content/class_distribution.png', dpi=150)
plt.show()


In [ ]:
# Visualize a few samples per class
fig, axes = plt.subplots(len(class_dirs), 4, figsize=(12, 3*len(class_dirs)))
for i, d in enumerate(class_dirs):
    imgs = glob.glob(os.path.join(d, '*.*'))[:4]
    for j, p in enumerate(imgs):
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        ax = axes[i, j] if len(class_dirs) > 1 else axes[j]
        ax.imshow(img); ax.axis('off')
        if j == 0: ax.set_ylabel(os.path.basename(d))
plt.tight_layout(); plt.savefig('/content/sample_grid.png', dpi=150); plt.show()


## Step 3 — Build a balanced, feasible subset

70k+ images is too much to train quickly on free Colab GPU time. We subsample a
**balanced** subset per class. This is a deliberate engineering decision — note it
in your report under "conflicting requirements: accuracy vs. computational cost".


In [ ]:
N_PER_CLASS = 110   # ~1,000 total images across classes -- good for assignment scope

subset_paths, subset_labels = [], []
for d in class_dirs:
    label = os.path.basename(d)
    imgs = glob.glob(os.path.join(d, '*.*'))
    random.shuffle(imgs)
    chosen = imgs[:N_PER_CLASS]
    subset_paths += chosen
    subset_labels += [label]*len(chosen)

print("Total images in working subset:", len(subset_paths))
le = LabelEncoder()
y_all = le.fit_transform(subset_labels)
print("Classes:", list(le.classes_))


## Step 4 — Preprocessing pipeline

Standard image-processing steps requested by the rubric: denoising, normalization,
edge detection. We define one function per step and show a before/after demo.


In [ ]:
IMG_SIZE = 224

def denoise(img):
    return cv2.fastNlMeansDenoisingColored(img, None, 7, 7, 7, 21)

def to_gray_edges(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(gray, 50, 150)
    return gray, edges

def normalize(img):
    return img.astype(np.float32) / 255.0

# Demo on one image
demo_path = random.choice(subset_paths)
demo_img = cv2.imread(demo_path)
demo_denoised = denoise(demo_img)
gray, edges = to_gray_edges(demo_denoised)

fig, ax = plt.subplots(1, 3, figsize=(12,4))
ax[0].imshow(cv2.cvtColor(demo_img, cv2.COLOR_BGR2RGB)); ax[0].set_title('Original'); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(demo_denoised, cv2.COLOR_BGR2RGB)); ax[1].set_title('Denoised'); ax[1].axis('off')
ax[2].imshow(edges, cmap='gray'); ax[2].set_title('Canny edges'); ax[2].axis('off')
plt.tight_layout(); plt.savefig('/content/preprocessing_demo.png', dpi=150); plt.show()


## Step 5 — Note localization (segmentation) → bounding box → IoU

We use classical thresholding + contour detection to find the note region and draw
a bounding box. This (a) crops out background clutter before classification, and
(b) gives us a predicted box we can compare to a rough "ground-truth" box (the full
frame, or a manually-marked box for a few sample images) using **IoU**, satisfying
the rubric's evaluation requirement.


In [ ]:
def localize_note(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        h, w = gray.shape
        return (0, 0, w, h)
    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)
    return (x, y, w, h)

def iou(boxA, boxB):
    xA, yA, wA, hA = boxA; xB, yB, wB, hB = boxB
    ax2, ay2 = xA+wA, yA+hA
    bx2, by2 = xB+wB, yB+hB
    ix1, iy1 = max(xA, xB), max(yA, yB)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    union = wA*hA + wB*hB - inter
    return inter/union if union > 0 else 0

# Demo: localization on a sample image, IoU vs full-frame reference box
img = cv2.imread(demo_path)
h, w = img.shape[:2]
pred_box = localize_note(img)
ref_box = (int(0.05*w), int(0.05*h), int(0.9*w), int(0.9*h))  # rough manual reference
print("Predicted box:", pred_box, "| IoU vs reference:", round(iou(pred_box, ref_box), 3))

x, y, bw, bh = pred_box
vis = img.copy()
cv2.rectangle(vis, (x,y), (x+bw, y+bh), (0,255,0), 4)
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.title('Localized note'); plt.axis('off')
plt.savefig('/content/localization_demo.png', dpi=150); plt.show()

# NOTE for report: to report a proper mean IoU, manually annotate bounding boxes
# for ~30-50 sample images (e.g. using https://www.makesense.ai/) and compare against
# localize_note() predictions. Keep it small and honest -- 30-50 images is enough
# to report a credible mean IoU without becoming a huge side-project.


## Step 6 — Classical feature extraction (HOG + color histogram)

These are the handcrafted descriptors the rubric asks for. We extract them for
every image in the subset to build the feature matrix for the classical ML models.


In [ ]:
def extract_features(img_path):
    img = cv2.imread(img_path)
    x, y, w, h = localize_note(img)
    crop = img[y:y+h, x:x+w] if w > 10 and h > 10 else img
    crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    hog_feat = hog(gray, orientations=9, pixels_per_cell=(16,16),
                    cells_per_block=(2,2), block_norm='L2-Hys')

    hist_feat = []
    for ch in range(3):
        h_ch = cv2.calcHist([crop], [ch], None, [32], [0,256]).flatten()
        h_ch = h_ch / (h_ch.sum() + 1e-6)
        hist_feat.append(h_ch)
    hist_feat = np.concatenate(hist_feat)

    return np.concatenate([hog_feat, hist_feat])

print("Extracting classical features for", len(subset_paths), "images (this takes a few minutes)...")
t0 = time.time()
X_classical = np.array([extract_features(p) for p in subset_paths])
print("Done in %.1fs | feature vector size: %d" % (time.time()-t0, X_classical.shape[1]))


## Step 7 — Train/test split (shared across classical models)

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c, paths_train, paths_test = train_test_split(
    X_classical, y_all, subset_paths, test_size=0.2, stratify=y_all, random_state=SEED)

scaler = StandardScaler()
X_train_c = scaler.fit_transform(X_train_c)
X_test_c = scaler.transform(X_test_c)
print(X_train_c.shape, X_test_c.shape)


## Step 8 — Classical ML models: SVM and Random Forest

Two classical baselines on the handcrafted features. This is one half of the
"classical vs deep learning" comparison that forms the paper's contribution.


In [ ]:
def evaluate_model(name, y_true, y_pred, results_dict):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    results_dict[name] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}
    print(f"{name}: acc={acc:.3f} precision={prec:.3f} recall={rec:.3f} f1={f1:.3f}")
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))
    return results_dict

results = {}

svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=SEED)
svm.fit(X_train_c, y_train_c)
pred_svm = svm.predict(X_test_c)
results = evaluate_model('SVM (HOG+ColorHist)', y_test_c, pred_svm, results)

rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
rf.fit(X_train_c, y_train_c)
pred_rf = rf.predict(X_test_c)
results = evaluate_model('RandomForest (HOG+ColorHist)', y_test_c, pred_rf, results)


In [ ]:
# Confusion matrix for the better classical model
best_classical_pred = pred_svm if results['SVM (HOG+ColorHist)']['f1'] >= results['RandomForest (HOG+ColorHist)']['f1'] else pred_rf
cm = confusion_matrix(y_test_c, best_classical_pred)
plt.figure(figsize=(7,6))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(len(le.classes_)), le.classes_, rotation=45)
plt.yticks(range(len(le.classes_)), le.classes_)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix — Best Classical Model')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j], ha='center', va='center', fontsize=8)
plt.colorbar(); plt.tight_layout()
plt.savefig('/content/cm_classical.png', dpi=150); plt.show()


## Step 9 — Deep learning pipeline: data loading + augmentation

Now the end-to-end CNN branch. We use `image_dataset_from_directory` directly on
the subset (copied into a clean folder structure) with augmentation, since CNNs
learn their own features rather than needing HOG/histograms as input.


In [ ]:
# Build a clean subset folder for Keras' directory loader
SUBSET_DIR = '/content/subset_data'
if os.path.exists(SUBSET_DIR):
    shutil.rmtree(SUBSET_DIR)
for p, lbl in zip(subset_paths, subset_labels):
    out_dir = os.path.join(SUBSET_DIR, lbl)
    os.makedirs(out_dir, exist_ok=True)
    shutil.copy(p, out_dir)

train_ds = tf.keras.utils.image_dataset_from_directory(
    SUBSET_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=32)
val_ds = tf.keras.utils.image_dataset_from_directory(
    SUBSET_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=32)

class_names = train_ds.class_names
print(class_names)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


## Step 10 — CNN model (transfer learning, MobileNetV2)

MobileNetV2 is chosen deliberately: it is lightweight and fast to fine-tune on
free-tier Colab, directly addressing the rubric's "model complexity vs.
computational feasibility" trade-off. The base is frozen first, then partially
unfrozen for fine-tuning.


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
EPOCHS_HEAD = 8
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD)


In [ ]:
# Fine-tune: unfreeze the top layers of the base model
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

EPOCHS_FT = 6
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT)


In [ ]:
# Plot training curves -- good figure for the report
def plot_history(h1, h2=None):
    acc = h1.history['accuracy'] + (h2.history['accuracy'] if h2 else [])
    val_acc = h1.history['val_accuracy'] + (h2.history['val_accuracy'] if h2 else [])
    plt.figure(figsize=(7,4))
    plt.plot(acc, label='train acc')
    plt.plot(val_acc, label='val acc')
    plt.axvline(len(h1.history['accuracy'])-0.5, color='gray', linestyle='--', label='fine-tune starts')
    plt.legend(); plt.title('CNN Training Curve'); plt.xlabel('epoch'); plt.ylabel('accuracy')
    plt.tight_layout(); plt.savefig('/content/cnn_training_curve.png', dpi=150); plt.show()

plot_history(history, history_ft)


## Step 11 — Evaluate the CNN with the same metrics as the classical models

In [ ]:
y_true_cnn, y_pred_cnn = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_pred_cnn += list(np.argmax(preds, axis=1))
    y_true_cnn += list(labels.numpy())

results = evaluate_model('CNN (MobileNetV2 transfer learning)', y_true_cnn, y_pred_cnn, results)

cm_cnn = confusion_matrix(y_true_cnn, y_pred_cnn)
plt.figure(figsize=(7,6))
plt.imshow(cm_cnn, cmap='Greens')
plt.xticks(range(len(class_names)), class_names, rotation=45)
plt.yticks(range(len(class_names)), class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix — CNN')
for i in range(cm_cnn.shape[0]):
    for j in range(cm_cnn.shape[1]):
        plt.text(j, i, cm_cnn[i,j], ha='center', va='center', fontsize=8)
plt.colorbar(); plt.tight_layout()
plt.savefig('/content/cm_cnn.png', dpi=150); plt.show()


## Step 12 — Classical vs Deep Learning: comparison table

This table is your headline "contribution" result — put it directly in the report.


In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.round(3)
print(comparison_df)
comparison_df.to_csv('/content/model_comparison.csv')
comparison_df


## Step 13 — Explainability: Grad-CAM on the CNN

Grad-CAM shows *which pixels* the CNN used to make its decision. This is a strong,
easy differentiator for your report/presentation — it visually proves the model is
looking at the note's design features rather than the background.


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), class_names[pred_index.numpy()]

# find the last conv layer name inside MobileNetV2 automatically
last_conv_name = None
for layer in base_model.layers[::-1]:
    if isinstance(layer, tf.keras.layers.Conv2D) or 'conv' in layer.name.lower():
        last_conv_name = layer.name
        break
print("Using layer:", last_conv_name)

sample_path = random.choice(subset_paths)
img = tf.keras.utils.load_img(sample_path, target_size=(IMG_SIZE, IMG_SIZE))
arr = tf.keras.utils.img_to_array(img)[np.newaxis, ...]

heatmap, pred_label = make_gradcam_heatmap(arr, model, last_conv_name)
heatmap_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))

fig, ax = plt.subplots(1, 2, figsize=(8,4))
ax[0].imshow(img); ax[0].set_title(f'Original (pred: {pred_label})'); ax[0].axis('off')
ax[1].imshow(img); ax[1].imshow(heatmap_resized, cmap='jet', alpha=0.5); ax[1].set_title('Grad-CAM'); ax[1].axis('off')
plt.tight_layout(); plt.savefig('/content/gradcam_demo.png', dpi=150); plt.show()


## Step 14 — Save everything you need for the report

Run this last. It saves the model and all metrics/figures to Google Drive so you
don't lose them when the Colab runtime disconnects.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/banknote_project_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(os.path.join(SAVE_DIR, 'banknote_cnn_model.keras'))
comparison_df.to_csv(os.path.join(SAVE_DIR, 'model_comparison.csv'))

for f in ['class_distribution.png', 'sample_grid.png', 'preprocessing_demo.png',
          'localization_demo.png', 'cm_classical.png', 'cnn_training_curve.png',
          'cm_cnn.png', 'gradcam_demo.png']:
    src = os.path.join('/content', f)
    if os.path.exists(src):
        shutil.copy(src, SAVE_DIR)

print("Saved everything to:", SAVE_DIR)
print("Files:", os.listdir(SAVE_DIR))


## Notes for the Report's Ethics Section

Don't skip this — it's an easy section to score well on if you're specific rather
than generic. Points to develop into full paragraphs:

- **Misclassification consequences**: this system is explicitly motivated by
  assistive use for visually impaired users. A misclassified note (e.g. 500 vs
  1000 BDT) has a direct real-world financial harm, unlike a typical toy image
  classification error. This raises the bar for acceptable error rate compared to,
  say, classifying cats vs dogs.
- **Data bias**: check your subset's class balance (`counts` above) and note
  whether the source images were captured under limited lighting/background/camera
  conditions — a model trained on clean, well-lit note photos may fail on worn,
  torn, or poorly lit notes in real use, which disproportionately affects
  lower-income or informal-economy users who handle more worn currency.
- **Privacy**: banknote images themselves are not personally identifying, but if
  deployed as a live camera app, the surrounding captured background could
  incidentally capture people, receipts, or personal spaces — worth a sentence on
  on-device processing / not storing raw camera frames.
- **Transparency**: explain how Grad-CAM in Step 13 supports transparency — it
  lets a developer/auditor verify the model is keying on the note's actual design
  features rather than spurious background cues.
